# SC-VAE — آموزش روی FFHQ واقعی (مورد ۱)

دادهٔ واقعیِ FFHQ به‌صورت **خودکار با کد** دانلود می‌شود (streaming از HuggingFace، نه کل ۷.۶ گیگ)،
سپس آموزشِ طولانی‌ترِ همان مسیرِ MSE، نمودار همگرایی، و ارزیابی روی **مجموعهٔ مجزای val**.

> ⚙️ قبل از اجرا: **Runtime → Change runtime type → GPU** (مثلاً T4).
> روشِ train/test همان کد اصلی است؛ فقط داده واقعی و آموزش طولانی‌تر شده.


## ۱) دریافت کد و نصب وابستگی‌ها

In [ ]:
!git clone https://github.com/e-shams-d/SC-VAE.git

In [ ]:
import os
os.chdir('/content/SC-VAE')
!pip install -q omegaconf easydict tensorboardX datasets

## ۲) دانلود خودکارِ FFHQ واقعی

از دیتاست `bitmind/ffhq-256` روی HuggingFace با **streaming** فقط چند صد تصویر می‌گیریم
(کل دیتاست ۷۰هزار تصویر / ۷.۶ گیگ است؛ ما فقط `N` تای اول را برمی‌داریم).

سپس به دو بخشِ **مجزای** train/val تقسیم می‌کنیم تا ارزیابی روی دادهٔ دیده‌نشده (تعمیم) معنادار باشد.


In [ ]:
import os, io
from PIL import Image as PILImage
from datasets import load_dataset

root = 'dataset/FFHQ/resized'
os.makedirs(root, exist_ok=True)

N = 512   # تعداد تصاویری که stream می‌کنیم (نه کل 70k)
ds = load_dataset("bitmind/ffhq-256", split="train", streaming=True)

names = []
for i, ex in enumerate(ds):
    if i >= N:
        break
    img = ex['image']
    if isinstance(img, dict):                 # بعضی نسخه‌ها {'bytes': ...} می‌دهند
        img = PILImage.open(io.BytesIO(img['bytes']))
    name = f'{i:05d}.jpg'
    img.convert('RGB').save(os.path.join(root, name))
    names.append(name)

# تقسیم مجزای train / val (val دیده‌نشده برای ارزیابی واقعی)
n_val = 64
val_names, train_names = names[:n_val], names[n_val:]
open('img_datasets/assets/ffhqtrain.txt', 'w').write('\n'.join(train_names) + '\n')
open('img_datasets/assets/ffhqvalidation.txt', 'w').write('\n'.join(val_names) + '\n')
print(f'saved {len(names)} FFHQ images -> {len(train_names)} train / {len(val_names)} val')

# (در صورت خطا در دیتاست بالا، جایگزین: load_dataset("merkol/ffhq-256", split="train", streaming=True))

## ۳) آموزش (طولانی‌تر، روی چهره‌های واقعی)

| پارامتر | مقدار | توضیح |
|---------|-------|-------|
| `experiment.epochs` | 30 | آموزش طولانی‌تر از دمو |
| `experiment.batch_size` | 4 | امن برای T4 (اگر OOM نشد، می‌توانی 8 بگذاری برای سرعت) |
| `optimizer.init_lr` | 2.0e-4 | + gradient clipping برای پایداری |
| `optimizer.warmup.min_lr` | 1.0e-5 | کاهش تدریجی LR |

⏱️ روی T4 حدوداً ۲۵–۳۵ دقیقه. برای سریع‌تر `epochs` را کم کن، برای کیفیت بهتر زیاد کن.
🆘 `CUDA out of memory` → `experiment.batch_size=4` را به `2` کم کن.


In [ ]:
!python main-stage1.py --dataset FFHQ \
  --model-config ./configs/ffhq/stage1/ffhq256-scvae16x16.yaml \
  --device cuda --num-workers 2 \
  experiment.epochs=30 experiment.batch_size=4 \
  optimizer.init_lr=2.0e-4 optimizer.warmup.min_lr=1.0e-5

## ۴) checkpointها + TensorBoard

In [ ]:
!ls -lh results/models/FFHQ/*/
%load_ext tensorboard
%tensorboard --logdir results/logs

## ۵) نمودار همگرایی

In [ ]:
import glob, os
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

log_dir = sorted(glob.glob('results/logs/FFHQ/*'), key=os.path.getmtime)[-1]
ea = EventAccumulator(log_dir, size_guidance={'scalars': 0})
ea.Reload()

def scal(tag):
    ev = ea.Scalars(tag)
    return [e.step for e in ev], [e.value for e in ev]

tags = ea.Tags()['scalars']
plt.figure(figsize=(8, 5))
if 'loss/train/reconstruction' in tags:
    st, vt = scal('loss/train/reconstruction')
    plt.plot(st, vt, alpha=0.25, label='train recon (per-batch)')
if 'loss/test/reconstruction' in tags:
    s, v = scal('loss/test/reconstruction')
    plt.plot(s, v, marker='o', linewidth=2, label='validation recon (per-epoch)')
plt.xlabel('training step'); plt.ylabel('reconstruction loss (MSE)')
plt.title('SC-VAE convergence (real FFHQ)'); plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('results/convergence.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved results/convergence.png')

## ۶) ارزیابی روی val دیده‌نشده (PSNR / SSIM / LPIPS + بازسازی)

ارزیابی روی همان ۶۴ تصویرِ مجزای val انجام می‌شود، پس اعداد **تعمیم** را نشان می‌دهند (نه حفظ‌کردن).


In [ ]:
!pip install -q piq lpips

import glob, os
ckpt = sorted(glob.glob('results/models/FFHQ/*/best.pt'), key=os.path.getmtime)[-1]
print('using checkpoint:', ckpt)

!python evaluate.py --dataset FFHQ \
  --model-config ./configs/ffhq/stage1/ffhq256-scvae16x16.yaml \
  --load-path "{ckpt}" --device cuda --batch-size 8

In [ ]:
from IPython.display import Image
Image('results/eval/reconstruction_compare.png')